# Data Preparation Workflow

This notebook triggers a SageMaker Processing Job to:
1. Enrich manifest with image URLs from Discogs API
2. Download all images
3. Upload everything to S3

## Prerequisites

- ✅ Input manifest uploaded to S3
- ✅ Discogs API credentials configured
- ✅ S3 bucket created

In [ ]:
# Install dependencies
!pip install sagemaker boto3 aiohttp pillow

In [ ]:
import boto3
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
import os
from datetime import datetime

# ============================================
# CONFIGURATION - UPDATE THESE VALUES
# ============================================
BUCKET_NAME = 'your-bucket'  # ← CHANGE THIS!
INPUT_MANIFEST = 's3://your-bucket/data/releases_manifest.jsonl'  # ← CHANGE THIS!
S3_PREFIX = 'data'
INSTANCE_TYPE = 'ml.m5.xlarge'  # CPU instance (cheaper)
REGION = 'us-east-1'

# Discogs API credentials
DISCOGS_KEY = os.getenv('DISCOGS_CONSUMER_KEY')  # Set in environment or Studio secrets
DISCOGS_SECRET = os.getenv('DISCOGS_CONSUMER_SECRET')

if not DISCOGS_KEY or not DISCOGS_SECRET:
    print("⚠ Warning: Discogs credentials not found!")
    print("Set DISCOGS_CONSUMER_KEY and DISCOGS_CONSUMER_SECRET environment variables")

# Get SageMaker execution role
role = sagemaker.get_execution_role()
print(f"✓ Using role: {role}")
print(f"✓ Bucket: {BUCKET_NAME}")
print(f"✓ Input manifest: {INPUT_MANIFEST}")

## Step 1: Upload Data Preparation Script

First, upload the `data_preparation.py` script to S3 or make it available in Studio.

In [ ]:
# Option 1: Upload script from local file
# Make sure data_preparation.py is in your Studio environment
# Or upload it to S3 first:

# Upload script to S3
s3_client = boto3.client('s3', region_name=REGION)
script_s3_path = f's3://{BUCKET_NAME}/code/data_preparation.py'

# If you have the script locally in Studio:
# s3_client.upload_file('data_preparation.py', BUCKET_NAME, 'code/data_preparation.py')
# print(f"✓ Script uploaded to {script_s3_path}")

# Option 2: Use script from S3 (if already uploaded)
script_s3_path = f's3://{BUCKET_NAME}/code/data_preparation.py'
print(f"Script location: {script_s3_path}")

## Step 2: Create and Run Processing Job

In [ ]:
# Create script processor
processor = ScriptProcessor(
    role=role,
    image_uri='763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.5.0-cpu-py311',
    command=['python3'],
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    sagemaker_session=sagemaker.Session(),
    base_job_name='data-prep',
)

# Generate job name
job_name = f'data-prep-{datetime.now().strftime("%Y%m%d-%H%M%S")}'

print(f"Starting processing job: {job_name}")
print(f"Instance type: {INSTANCE_TYPE}")
print(f"This will take 10-60 minutes depending on dataset size...")

# Run processing job
processor.run(
    code='data_preparation.py',  # Local file in Studio, or use S3 path
    inputs=[
        ProcessingInput(
            source=INPUT_MANIFEST,
            destination='/opt/ml/processing/input',
            input_name='manifest'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{BUCKET_NAME}/{S3_PREFIX}/',
            output_name='output'
        )
    ],
    arguments=[
        '--s3-bucket', BUCKET_NAME,
        '--s3-prefix', S3_PREFIX,
        '--region', REGION,
    ],
    job_name=job_name,
    environment={
        'DISCOGS_CONSUMER_KEY': DISCOGS_KEY,
        'DISCOGS_CONSUMER_SECRET': DISCOGS_SECRET,
    },
    wait=False,  # Don't wait, return immediately
)

print(f"\n✓ Processing job started!")
print(f"Job name: {job_name}")
print(f"\nMonitor progress:")
print(f"  Console: https://console.aws.amazon.com/sagemaker/home#/processing-jobs/{job_name}")
print(f"  CLI: aws sagemaker describe-processing-job --processing-job-name {job_name}")

In [ ]:
# Check job status
import time

sagemaker_client = boto3.client('sagemaker', region_name=REGION)

# Wait and check status
print("Checking job status...")
for i in range(60):  # Check for up to 60 minutes
    response = sagemaker_client.describe_processing_job(ProcessingJobName=job_name)
    status = response['ProcessingJobStatus']
    
    print(f"[{i+1}] Status: {status}")
    
    if status in ['Completed', 'Failed', 'Stopped']:
        break
    
    time.sleep(60)  # Wait 1 minute between checks

# Final status
response = sagemaker_client.describe_processing_job(ProcessingJobName=job_name)
print(f"\nFinal status: {response['ProcessingJobStatus']}")

if response['ProcessingJobStatus'] == 'Completed':
    print("✓ Job completed successfully!")
    print(f"\nOutput location: s3://{BUCKET_NAME}/{S3_PREFIX}/")
    print(f"  - releases_manifest_enriched.jsonl")
    print(f"  - images/")
else:
    print("✗ Job failed or stopped")
    print("Check CloudWatch logs for details")

## Step 4: Verify Output

Check that files were created in S3.

In [ ]:
# List output files
s3_client = boto3.client('s3', region_name=REGION)

print("Checking S3 output...")
try:
    # List files in output directory
    response = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix=f'{S3_PREFIX}/'
    )
    
    if 'Contents' in response:
        print(f"\n✓ Found {len(response['Contents'])} files:")
        for obj in response['Contents'][:10]:  # Show first 10
            print(f"  - {obj['Key']}")
        
        if len(response['Contents']) > 10:
            print(f"  ... and {len(response['Contents']) - 10} more")
        
        # Check for enriched manifest
        manifest_files = [obj for obj in response['Contents'] if 'manifest_enriched' in obj['Key']]
        if manifest_files:
            print(f"\n✓ Enriched manifest found: {manifest_files[0]['Key']}")
        
        # Count images
        image_files = [obj for obj in response['Contents'] if 'images/' in obj['Key']]
        if image_files:
            print(f"✓ Found {len(image_files)} images")
    else:
        print("⚠ No files found in output directory")
        print("Job may still be running or failed")
        
except Exception as e:
    print(f"Error checking S3: {e}")

## Next Steps

After data preparation completes:

1. ✅ Verify output in S3
2. ✅ Use enriched manifest for training (see `notebooks/studio_notebook.ipynb`)
3. ✅ Train model
4. ✅ Deploy endpoint

See `docs/DATA_PREPARATION_WORKFLOW.md` for more details and troubleshooting.